# Zhuang, Lemak, Sridhar & Nelson (2025): what DANDI:001177 and the paper's Zenodo deposit support

Paper: Zhuang X, Lemak J, Sridhar S, Nelson AB. "Aberrant striatal firing mediates impulsive
decision-making in a mouse model of Parkinson's disease." Brain (2025), doi:10.1093/brain/awaf312.
The claims file (`claims/001177/10.1093-brain-awaf312.yaml`) was written from the version of record
(Europe PMC full text, PMC12597312) with its supplement, and checked by a second reader.

Data.

- DANDI:001177, version 0.250902.2127, streamed. The paper cites the dandiset without a version
  ("Raw in vivo electrophysiology datasets have been deposited in DANDI
  (https://dandiarchive.org/dandiset/001177)"). This is the only published version; the draft was
  modified later (2026-09-03) with the same 56 assets and total size, so the published version is
  pinned, as the protocol asks when the paper names none. The dandiset holds the in vivo
  electrophysiology of Figures 4, 6F-K, S5 and S8.
- Zenodo record 15258978 (doi:10.5281/zenodo.15258978), the paper's deposit of its behaviour,
  histology, rotarod and slice data, which the claims file names for every `data: elsewhere`
  claim. It has one version. Its files are streamed into memory.
- The paper's analysis code, UCSF-Nelson-Lab/ICD-Cognitive-Analysis at commit 5528e222, was read for
  the event names and the curve fit; it is not run (it needs a GUI), and `icd.py` re-implements
  the parts used, citing them.

| Claims | Data | Status here |
|---|---|---|
| 14 electrophysiology claims (Figure 4, S5, 6F-I, S8) | DANDI:001177 | blocked: the dandiset has no sorted, classified or optically identified units, no laser pulse times and no injection time |
| 30 behaviour, histology, rotarod and slice claims | Zenodo 15258978 | 28 tested; two blocked (sex not recorded; no PPX open field file) |
| figs2abc-sex-similar | Zenodo | blocked: the deposit does not give each mouse's sex |
| figs1ef-ppx-locomotion | Zenodo | blocked: no PPX open field file in the deposit |

Each claim's outcome is in the attempt record and the summary at the end.

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import icd

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
print("DANDI", icd.DANDISET, icd.DANDI_VERSION, "| Zenodo", icd.ZENODO_DOI, "| code", icd.CODE_COMMIT[:8])

## Data check

DANDI:001177. Every file is opened and its metadata, groups and array shapes are listed (no
voltage is read). The claims need, per session, sorted single units, their putative MSN class,
their optical identification (laser pulse times) and the PPX injection time.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

meta = icd.dandi_metadata()
assets = icd.dandi_assets()
print(meta["name"], "|", meta["version"], "|", len(assets), "assets |", round(assets.size_gb.sum()), "GB")
with ThreadPoolExecutor(8) as ex:
    survey = pd.DataFrame(list(ex.map(icd.survey_file, assets.asset_id)))
survey.insert(0, "path", assets.path.str.split("/").str[-1].str.replace("_ecephys.nwb", ""))
print(survey[["path", "subject", "genotype", "sex", "lesion", "start", "segments_min", "n_channels",
              "n_spike_event_series", "n_threshold_crossings", "has_units", "n_units"]].to_string())

In [ ]:
subj = survey.drop_duplicates("subject")
print("subjects:", len(subj), "| by lesion:", subj.lesion.value_counts().to_dict(),
      "| by genotype:", subj.genotype.str.replace("-", "").value_counts().to_dict())
print("sessions by kind:", survey.path.str.contains("Saline").map({True: "saline", False: "PPX"}).value_counts().to_dict())
print("saline-session mice:", sorted(survey[survey.path.str.contains("Saline")].subject))
print("files with a units table:", survey[survey.has_units].path.tolist())
print("units table columns:", survey[survey.has_units].units_columns.unique().tolist())
for _, r in survey[survey.has_units].iterrows():
    c = np.array(r.unit_spike_counts)
    print(f"  {r.path}: {len(c)} units, spike counts {c.min()} to {c.max()}; even-numbered units median "
          f"{np.median(c[::2]):.0f}, odd-numbered median {np.median(c[1::2]):.0f}")
print("files with intervals, stimulus or optogenetics metadata:",
      int((survey.intervals != "").sum()), int((survey.stimulus != "").sum()), int(survey.ogen.sum()))
print("processing modules:", survey.processing.unique().tolist())
print("session start years:", survey.start.str[:4].value_counts().sort_index().to_dict())
print("description encoding (first 40 characters):", survey.description_start.iloc[0])

What the dandiset holds: 18 mice, 7 with striatal saline and 11 with 6-OHDA (as the paper's Ctrl
N = 7 and Park N = 11), 52 PPX open field sessions and 4 saline sessions (4 mice, as Figure S5E),
each a 30 kHz 32-channel recording split into segments (most often about 30 min, then about 300 min,
then about 70 min, the last presumably the laser session), the 1 kHz LFP, and per electrode the
threshold crossings with their 48-sample waveforms ("Events detected with threshold crossing,
manually set for each electrode"). These are the input to the paper's manual sorting in Plexon
Offline Sorter, not its output. Two files carry a `units` table, with spike times only; in both,
even-numbered units have far more spikes than odd-numbered ones, consistent with an unsorted and a
sorted cluster per channel, but no column says which, nor the channel, the MSN class or the
optical label. No file has laser pulse times, an injection time or behaviour. The segment gaps are
consistent with an injection after the first 30 min, but this is not stated in the files.
No session has a trial or event table, so whether the streams share a trial clock cannot be
established for any electrophysiology session; this is moot here, as no claim on them is tested.

Zenodo deposit: the files, then the delay discounting sessions parsed from the raw operant logs.

In [ ]:
files = icd.zenodo_files()
print(files.to_string())
groups = icd.readme_groups()
print({f"{e}/{g}": len(v) for (e, g), v in groups.items()})
sess = icd.sessions()
print("skipped files:", sess.attrs["skipped"])
print(sess.drop_duplicates(["experiment", "file"]).groupby(["experiment", "group", "condition"]).agg(
    mice=("mouse", "nunique"), sessions=("file", "nunique")).to_string())
print("mice per experiment and group with no drug session:",
      sess.groupby(["experiment", "group", "mouse"]).condition.apply(lambda c: "on" not in set(c))
      .loc[lambda s: s].index.tolist())
import hashlib
digests = {e: {n.split("/")[-1]: hashlib.md5(z.read(n)).hexdigest() for n in z.namelist()
               if n.upper().endswith(".CSV") and not n.startswith("__")} for e, z in icd.cognitive_archives().items()}
for a in digests:
    for b in digests:
        if a < b:
            common = set(digests[a]) & set(digests[b])
            print(f"files in both {a} and {b}: {len(common)}, byte-identical {sum(digests[a][f] == digests[b][f] for f in common)}")
listed = {(e, m) for (e, g), ids in groups.items() for m in ids}
present = set(zip(sess.experiment, sess.mouse))
print("README mice with no delay discounting file:", sorted(listed - present))
print("free choice trials per block:", sess.trials.value_counts().sort_index().to_dict())
print("block labels (delay -> label: sessions):",
      sess.groupby(["delay", "block_label"]).file.nunique().to_dict())
print("sessions dated from the log (file name malformed):",
      sess[sess.date_from == "log"].drop_duplicates("file")[["file", "date"]].values.tolist())
print("range of % delayed/large:", sess.pct_large.min(), sess.pct_large.max(),
      "| latency (s):", round(np.nanmin(sess.lat_all), 3), round(np.nanmax(sess.lat_all), 3))

Clock and block check. In every free choice block, the delay between the side nosepoke and the
large reward delivery, measured on the logs' own clock, should equal the block's delay. This tests
the block-to-delay mapping and that the event times share one clock.

In [ ]:
def reward_delays(text):
    out, block, poke = {}, None, None
    for line in text.splitlines()[1:]:
        p = [x.strip().strip('"') for x in line.split(",")]
        if len(p) < 3:
            continue
        m = icd.re.match(r"B(\d+)_", p[1])
        if m:
            block = icd.FREE_BLOCKS.get("B" + m.group(1))
        elif block is not None and p[1] in ("np_B in", "np_D in") and poke is None:
            poke = icd._ms(p[2])
        elif p[1] == "trial start" or p[1] == "cen tri":
            poke = None
        elif block is not None and p[1] == "l reward delivery" and poke is not None:
            out.setdefault(block, []).append((icd._ms(p[2]) - poke) / 1000)
    return out

rows = []
for exp, zf in icd.cognitive_archives().items():
    for name in zf.namelist():
        base = name.split("/")[-1]
        if "_delay" in base and not name.startswith("__"):
            for d, v in reward_delays(zf.read(name).decode("latin-1")).items():
                rows.append({"experiment": exp, "file": base, "delay": d, "median_s": np.median(v),
                             "old_labels": base.startswith("P")})
rd = pd.DataFrame(rows)
print(rd.groupby(["experiment", "old_labels", "delay"]).median_s.describe()[["count", "25%", "50%", "75%"]].round(2))

The measured nosepoke-to-reward delays are about 0, 3, 6 and 9 s in the four free choice blocks of
every experiment, so the blocks are the paper's delays and the logs keep one clock. The exception
is the 2020 cohort of six control PPX mice (files of mice P1 to P7), whose block labels name delays
of 0, 6, 12 and 18 s and whose measured delays agree with those labels; 30 of their sessions have
no fourth block. The paper's code counts these blocks as the other mice's; so does this analysis,
which is a deviation noted under each affected claim. Sessions of the 2021 cohort (C mice) have 30
free choice trials per block, the Methods 20.

Shared definitions for the delay discounting claims. Per mouse and condition, the per-delay value
is the mean over the condition's sessions. Readings, from the claims file:

- baseline (a) every baseline session in the deposit, (b) the last stable run by the Methods' CV
  rule;
- drug ON (a) the first ON session by date, (b) the mean over the first four ON sessions;
- drug OFF: every washout session (the claims file gives no reading; all are used, as
  Summary_Cognitive.m averages a mouse's sessions within a condition), with the first OFF session
  as a variant.

In [ ]:
W = icd.wide(sess)
D = icd.DELAYS

def per_mouse(exp, group, cond, reading, value="pct_large"):
    w = (icd.wide(sess, value) if value != "pct_large" else W)
    w = w[(w.experiment == exp) & (w.group == group) & (w.condition == cond)].sort_values("date")
    out = {}
    for m, g in w.groupby("mouse"):
        if reading == "all":
            sel = g
        elif reading == "first":
            sel = g.head(1)
        elif reading == "four":
            sel = g.head(4)
        elif reading == "fourth":
            sel = g.iloc[3:4]
        elif reading == "stable":
            later = W[(W.experiment == exp) & (W.mouse == m) & (W.condition != "baseline")].date.min()
            files_b = W[(W.experiment == exp) & (W.mouse == m) & (W.condition == "baseline")]
            files_b = files_b[files_b.date < later] if pd.notna(later) else files_b
            sel = g[g.file.isin(icd.stable_window(files_b))]
        if len(sel):
            out[m] = sel[D].mean()
    t = pd.DataFrame(out).T
    t.index.name = "mouse"
    return t

def long(tab, **kw):
    l = tab.reset_index().melt(id_vars="mouse", var_name="delay", value_name="v")
    for k, v in kw.items():
        l[k] = v
    return l

def drug_test(exp, group, rb, rd, value="pct_large"):
    base, on = per_mouse(exp, group, "baseline", rb, value), per_mouse(exp, group, "on", rd, value)
    L = pd.concat([long(base, t="baseline"), long(on, t="on")])
    return icd.split_plot_anova(L, "v", "t", ["baseline", "on"])

def show(res, label):
    print(f"{label}: mice {res['n_mice']}, units {res['n_units']}, treatment F{res['treat_df']} = "
          f"{res['treat_F']:.3f}, p = {res['treat_p']:.3g}; delay F{res['delay_df']} = {res['delay_F']:.2f}, "
          f"p = {res['delay_p']:.3g}; interaction p = {res['inter_p']:.3g}")
    print(res["per_delay"].round(4).to_string(index=False))

def params(tab):
    r = {m: icd.fit_curve(row.values / 100) for m, row in tab.iterrows()}
    return pd.DataFrame(r, index=["A", "K", "AUC"]).T

## fig1fg-baseline-similar

> PPX-naive control and parkinsonian mice showed similar delay discounting.

Criterion: two-way ANOVA on group and delay (Table 1), Bonferroni per delay; reproduced when the
group effect and every per-delay comparison have p >= 0.05.

In [ ]:
baseline_res = {}
for r in ["all", "stable"]:
    b = {g: per_mouse("PPX", g, "baseline", r) for g in ["control", "parkinsonian"]}
    L = pd.concat([long(b[g], grp=g) for g in b])
    res = icd.between_anova(L, "v", "grp")
    baseline_res[r] = res
    print(f"reading {r}: mice {res['n_mice']}; group F{res['df']} = {res['factor_F']:.4f}, p = {res['factor_p']:.4f}; "
          f"delay F{res['delay_df']} = {res['delay_F']:.2f}, p = {res['delay_p']:.3g}; interaction p = {res['inter_p']:.3f}")
    print(res["per_delay"].round(3).to_string(index=False))
nostable = [m for g in ["control", "parkinsonian"] for m in per_mouse("PPX", g, "baseline", "all").index
            if m not in per_mouse("PPX", g, "baseline", "stable").index]
print("mice with no stable run under the CV rule:", len(nostable))

**Result: reproduced.** Under reading (a), every baseline session (16 control and 31 parkinsonian
mice), the group effect is F = 0.007, p = 0.93 (paper F = 0.003, p = 0.96) and no per-delay
comparison differs (all Bonferroni p = 1). Reading (b) keeps only the 12 mice whose deposited
baseline sessions pass the CV rule; it also finds no group effect (p = 0.11) and no per-delay
difference. The two control mice of the 2020 cohort without a fourth block are missing at 9 s.

## fig1fg-delay-discounting

> At baseline both control and parkinsonian mice chose the delayed/large reward less often as its
> delay increased.

Criterion: the ANOVA above; reproduced when the delay effect has p < 0.05 and in each group the
mean is lower at 9 s than at 0 s.

In [ ]:
for r, res in baseline_res.items():
    pdl = res["per_delay"].set_index("delay")
    print(f"reading {r}: delay F{res['delay_df']} = {res['delay_F']:.2f}, p = {res['delay_p']:.3g}; "
          f"control {pdl.loc[0, 'control']:.1f} -> {pdl.loc[9, 'control']:.1f} %, "
          f"parkinsonian {pdl.loc[0, 'parkinsonian']:.1f} -> {pdl.loc[9, 'parkinsonian']:.1f} % (0 s -> 9 s)")
fig, ax = plt.subplots(figsize=(4, 3))
for g in ["control", "parkinsonian"]:
    t = per_mouse("PPX", g, "baseline", "all")
    ax.errorbar(D, t.mean(), t.sem(), marker="o", label=g)
ax.set(xlabel="delay (s)", ylabel="% delayed/large", ylim=(0, 100), title="Baseline, reading (a)")
ax.legend()
plt.show()

**Result: reproduced.** Under reading (a) the delay effect is F(3, 178) = 38.15, p = 4e-19 (paper
F(3, 180) = 38.53), and the choice of the delayed/large reward falls from 91.7 to 32.1% in control
and from 85.7 to 32.9% in parkinsonian mice between 0 and 9 s. Under reading (b) the means also
fall (95.4 to 82.4%, 95.7 to 58.8%) but the delay effect is not significant (p = 0.11) in 12 mice.

## fig1f-ctrl-ppx-no-change and fig1g-park-ppx-fewer-large

> PPX did not change delay discounting in healthy control mice.
> In parkinsonian mice PPX reduced the likelihood of choosing the delayed/large reward at every delay.

Split-plot ANOVA (unit = mouse x delay, see `icd.py`) of baseline against PPX ON, Bonferroni per
delay on the pooled error (paired t tests as a variant). Readings: baseline (a)/(b) x PPX first/four.

In [ ]:
for rb in ["all", "stable"]:
    for rd in ["first", "four"]:
        show(drug_test("PPX", "control", rb, rd), f"control, baseline {rb}, PPX {rd}")

**Result: reproduced.** In the 16 control mice the PPX effect is not significant under any reading
(p = 0.10 to 0.38; paper F(1, 60) = 0.96, p = 0.33), and no per-delay comparison reaches
p < 0.05 on the pooled error. With the first four PPX sessions and every baseline, F(1, 54) = 1.98,
p = 0.16. Six control mice of the 2020 cohort had delays of 0, 6, 12 and 18 s; their fourth block is
missing in some sessions, so 10 mice enter at 9 s.

In [ ]:
for rb in ["all", "stable"]:
    for rd in ["first", "four"]:
        show(drug_test("PPX", "parkinsonian", rb, rd), f"parkinsonian, baseline {rb}, PPX {rd}")

**Result: reproduced.** With every baseline session and the mean of the four PPX sessions, the PPX
effect is F(1, 116) = 91.46, p = 2.5e-16 (paper F(1, 120) = 95.48) and the delayed/large choices are
lower under PPX at all four delays (Bonferroni p < 0.0001, < 0.0001, 0.0002 and 0.022 at 0, 3, 6
and 9 s; paired variant p <= 0.0034). With the first PPX session only, two of four delays are
significant on the pooled error (three with paired tests). 30 of the 31 parkinsonian mice have PPX sessions in the deposit (mouse E6 has only
baseline sessions).

## fig1ijk-ppx-a-k-auc

> In parkinsonian mice PPX decreased A, increased K and reduced the area under the discounting curve.

Fit of V = A/(1 + K D) per mouse and condition, Friedman over baseline, PPX ON and PPX OFF, Dunn's
comparisons against baseline (Bonferroni over two).

In [ ]:
def friedman_block(exp, group, ron, roff, label):
    P = {"baseline": params(per_mouse(exp, group, "baseline", "all")),
         "on": params(per_mouse(exp, group, "on", ron)),
         "off": params(per_mouse(exp, group, "off", roff))}
    fails = {k: int(v.A.isna().sum()) for k, v in P.items()}
    out = {}
    for p in ["A", "K", "AUC"]:
        t = pd.DataFrame({k: v[p] for k, v in P.items()})
        r = icd.friedman_dunn(t, "baseline")
        rm = icd.rm_oneway(t, "baseline")
        out[p] = (r, rm, t)
        print(f"{label} {p}: n = {r['n']}, Friedman chi2 = {r['chi2']:.2f}, p = {r['p']:.3g}; "
              + "; ".join(f"Dunn baseline vs {c}: median {v['median_ref']:.3f} -> {v['median']:.3f}, p = {v['p_bonf']:.3g}"
                          for c, v in r["dunn"].items())
              + f" | RM ANOVA p = {rm['p']:.3g}, Bonferroni " + ", ".join(f"{k} p = {v:.3g}" for k, v in rm["pairs"].items()))
    print(f"{label}: failed fits per condition {fails}")
    return out

In [ ]:
ppx_params = {}
for ron, roff in [("four", "all"), ("first", "first")]:
    ppx_params[ron] = friedman_block("PPX", "parkinsonian", ron, roff, f"PPX ON {ron}, OFF {roff}")

**Result: partially reproduced.** With the four PPX sessions, A falls (median 0.891 to 0.785, Dunn
p = 1.9e-06) and AUC falls (3.95 to 2.91, p = 2.7e-08), but the rise in K (0.258 to 0.321) is not
significant by Dunn's test (p = 0.19), although the Friedman test over the three conditions gives
p = 0.032 (the paper reports p = 0.03 for K). With the first PPX session, A and AUC fall (p = 0.012
and 5.3e-05) and K does not change (p = 1). 30 mice enter; no fit failed.

## figs2ijk-ppx-washout

> After a 48 h PPX washout the A value of parkinsonian mice returned to baseline while the changes in
> K and AUC persisted.

The same fits; readings (a) Friedman with Dunn's, (b) repeated measures one-way ANOVA with Bonferroni
(Table 1). Reproduced when baseline against OFF has p >= 0.05 for A and p < 0.05 for K and AUC, in
the direction seen under PPX.

In [ ]:
for ron, out in ppx_params.items():
    for p in ["A", "K", "AUC"]:
        r, rm, t = out[p]
        print(f"ON {ron} {p}: baseline vs OFF Dunn p = {r['dunn']['off']['p_bonf']:.3g} "
              f"(median {r['dunn']['off']['median_ref']:.3f} -> {r['dunn']['off']['median']:.3f}); "
              f"RM ANOVA Bonferroni p = {rm['pairs']['baseline vs off']:.3g}; ON vs OFF p = {rm['pairs']['on vs off']:.3g}")

**Result: partially reproduced.** With every washout session, A after washout does not differ from
baseline (Dunn p = 0.73, RM ANOVA p > 0.05) and K stays raised (0.258 to 0.419, Dunn p = 0.020), but
AUC after washout (3.95 to 3.38) does not differ from baseline at p < 0.05 by Dunn's test
(p = 0.078). Under reading (b), repeated measures ANOVA with Bonferroni, AUC persists (p = 0.039)
and K does not (p = 0.063). Neither reading gives all three parts.

## figs2de-omissions-low

> Omission rates were low at baseline and under PPX; per-delay comparisons found no PPX effect on
> center or side omissions in either group, except center omissions at the 6 s delay in
> parkinsonian mice.

In [ ]:
om = {}
for value in ["pct_center_om", "pct_side_om"]:
    for g in ["control", "parkinsonian"]:
        res = drug_test("PPX", g, "all", "four", value)
        om[(value, g)] = res
        show(res, f"{value}, {g}")
        base = per_mouse("PPX", g, "baseline", "all", value)
        on = per_mouse("PPX", g, "on", "four", value)
        print(f"   mean rate baseline {np.nanmean(base.values):.2f} %, PPX {np.nanmean(on.values):.2f} %")

**Result: reproduced.** With the four PPX sessions, center omissions of parkinsonian mice
differ only at 6 s (4.5 against 11.7%, Bonferroni p < 0.0001; other delays p = 0.15 to 1), with a
PPX main effect (F(1, 116) = 24.82, p = 2.2e-06; paper F(1, 120) = 16.54); no side omission
comparison and no control comparison is significant (smallest p = 0.60). Mean omission rates are
5.0 and 7.8% (center) and 1.6 and 1.9% (side) of free choice trials in parkinsonian mice at
baseline and on PPX.

## figs2gh-ppx-latency

> In parkinsonian mice on PPX, the latency to choose the immediate/small reward was shorter than at
> baseline, and the modulation of choice latency by outcome was lost.

In [ ]:
for rd in ["four", "first"]:
    show(drug_test("PPX", "parkinsonian", "all", rd, "lat_small"), f"latency to small reward, parkinsonian, baseline vs PPX {rd}")
for cond, rd in [("baseline", "all"), ("on", "four"), ("on", "first")]:
    lg = per_mouse("PPX", "parkinsonian", cond, rd, "lat_large")
    sm = per_mouse("PPX", "parkinsonian", cond, rd, "lat_small")
    L = pd.concat([long(lg, size="large"), long(sm, size="small")])
    show(icd.split_plot_anova(L, "v", "size", ["large", "small"]), f"reward size effect on latency, {cond} {rd}")

**Result: partially reproduced.** The latency to choose the small reward is shorter on PPX at
0 and 3 s (1.28 to 1.09 s and 1.38 to 1.02 s) but the PPX effect is not significant (F(1, 110) =
2.87, p = 0.093 with four sessions; p = 0.15 with the first; paper F(1, 114) = 4.505, p = 0.02).
The reward size effect on latency is present at baseline (F(1, 114) = 5.33, p = 0.023; paper
F(1, 114) = 7.016, p = 0.0092) and absent on PPX (p = 0.71 with four sessions, 0.19 with the first;
paper p = 0.70), as the criterion requires.

## A77636 claims: fig2d-ctrl-a77636-no-change, fig2e-park-a77636-fewer-large, fig2fgh-a77636-k-auc, figs3-a77636-washout-latency

In [ ]:
for rb in ["all", "stable"]:
    for rd in ["first", "four"]:
        show(drug_test("A77636", "control", rb, rd), f"A77636 control, baseline {rb}, ON {rd}")

**Result: reproduced.** In the 10 control mice the A77636 effect is not significant under
any reading (p = 0.31 to 0.80; paper F(1, 36) = 1.05, p = 0.31) and no per-delay comparison
reaches p < 0.05.

In [ ]:
for rb in ["all", "stable"]:
    for rd in ["first", "four"]:
        show(drug_test("A77636", "parkinsonian", rb, rd), f"A77636 parkinsonian, baseline {rb}, ON {rd}")

**Result: reproduced.** With every baseline session and the four A77636 sessions, the
effect is F(1, 60) = 45.06, p = 7.7e-09 (paper F(1, 60) = 45.68) and the choices are lower at 3, 6
and 9 s (Bonferroni p = 0.0049, 0.0003, 0.0003; paper 0.0047, 0.0003, 0.0002), not at 0 s
(p = 0.56; paper 0.57).

In [ ]:
a77_params = {}
for ron, roff in [("four", "all"), ("first", "first")]:
    a77_params[ron] = friedman_block("A77636", "parkinsonian", ron, roff, f"A77636 ON {ron}, OFF {roff}")

**Result: reproduced.** With the four A77636 sessions, A does not change (Friedman
p = 0.31, Dunn p = 0.31; paper 0.31), K rises (median 0.115 to 0.291, Dunn p = 0.0053; paper
0.0053) and AUC falls (5.82 to 3.97, Dunn p = 0.0008; paper 0.0008). With the first session only,
the K and AUC changes are not significant (Dunn p = 0.43 and 0.10).

In [ ]:
for ron, out in a77_params.items():
    for p in ["K", "AUC"]:
        r, rm, t = out[p]
        on_off = stats.wilcoxon(t.dropna()["on"], t.dropna()["off"]).pvalue
        k = t.dropna()
        ranks = k.rank(axis=1)
        z = (ranks["off"].mean() - ranks["on"].mean()) / np.sqrt(3 * 4 / (6 * len(k)))
        print(f"ON {ron} {p}: medians baseline {k.baseline.median():.3f}, ON {k.on.median():.3f}, OFF {k.off.median():.3f}; "
              f"Dunn ON vs OFF p = {min(1, 2 * stats.norm.sf(abs(z)) * 3):.3g} (three comparisons); "
              f"Dunn baseline vs OFF p = {r['dunn']['off']['p_bonf']:.3g}")
for value in ["pct_center_om", "pct_side_om"]:
    show(drug_test("A77636", "parkinsonian", "all", "four", value), f"A77636 parkinsonian, {value}")
for cond, rd in [("baseline", "all"), ("on", "four")]:
    lg = per_mouse("A77636", "parkinsonian", cond, rd, "lat_large")
    sm = per_mouse("A77636", "parkinsonian", cond, rd, "lat_small")
    L = pd.concat([long(lg, size="large"), long(sm, size="small")])
    show(icd.split_plot_anova(L, "v", "size", ["large", "small"]), f"A77636 reward size effect on latency, {cond}")

**Result: partially reproduced.** After washout K (0.209) and AUC (4.93) do not differ from
the A77636 values (Dunn p = 1 and 0.65) and stay on the same side of baseline (0.115, 5.82; Dunn
baseline against OFF p = 0.043 for both). No per-delay omission comparison is significant
(smallest Bonferroni p = 0.65). The reward size effect on choice latency is significant at
baseline (F(1, 52) = 14.70, p = 0.0003; paper F(1, 52) = 11.70) but, against the criterion, also
on A77636 (F(1, 56) = 4.84, p = 0.032; paper p = 0.07).

## DREADD claims: fig5g-mcherry-no-change, fig5hi-dreadd-fewer-large, fig5jl-dreadd-params

Pre-CNO baseline sessions against the mean of the post-CNO sessions (first four), per group.

In [ ]:
for rb in ["all", "stable"]:
    show(drug_test("DREADD", "mCherry", rb, "four"), f"mCherry, baseline {rb}, CNO four")

**Result: reproduced.** In the 14 mCherry mice with delay discounting files, CNO has no
effect (F(1, 52) = 0.94, p = 0.34; paper F(1, 56) = 0.97, p = 0.33) and no per-delay comparison
differs (all Bonferroni p = 1). The README lists 15 mCherry mice; R5.5 has no delay discounting
file in the deposit.

In [ ]:
for g in ["hM4Di", "hM3Dq"]:
    for rb in ["all", "stable"]:
        show(drug_test("DREADD", g, rb, "four"), f"{g}, baseline {rb}, CNO four")

**Result: partially reproduced.** In hM3Dq mice CNO lowers the delayed/large choices
(F(1, 44) = 40.94, p = 8.8e-08; paper 42.30) at 3, 6 and 9 s (p = 0.0006, 0.0013, 0.026). In hM4Di
mice the main effect holds (F(1, 44) = 12.77, p = 0.0009; paper 22.22) but only the 3 s comparison
is significant (p = 0.022; 6 s p = 0.38, 9 s p = 0.62).

In [ ]:
for g in ["mCherry", "hM4Di", "hM3Dq"]:
    base = params(per_mouse("DREADD", g, "baseline", "all"))
    on = params(per_mouse("DREADD", g, "on", "four"))
    for p in ["A", "K", "AUC"]:
        t = pd.DataFrame({"baseline": base[p], "on": on[p]}).dropna()
        w = stats.wilcoxon(t.baseline, t.on).pvalue
        # reading (b): Friedman over the two conditions, which reduces to a two-sided sign test
        fr = stats.binomtest(int((t.on > t.baseline).sum()), int((t.on != t.baseline).sum())).pvalue
        print(f"{g} {p}: n = {len(t)}, median {t.baseline.median():.3f} -> {t.on.median():.3f}, "
              f"higher in {int((t.on > t.baseline).sum())}; Wilcoxon p = {w:.3g}; Friedman (sign test) p = {fr:.3g}")
    print(f"   failed fits: baseline {int(base.A.isna().sum())}, CNO {int(on.A.isna().sum())}")

**Result: partially reproduced.** Under reading (a), Wilcoxon (Table 1), hM3Dq mice show
the three changes (A 0.923 to 0.880, p = 0.012; K 0.133 to 0.198, p = 0.0015; AUC 5.10 to 4.31,
p = 0.0005), hM4Di mice a lower AUC (6.08 to 5.78, p = 0.043) and unchanged A (p = 0.20), and
mCherry mice no change (p >= 0.36). The rise in K of hM4Di mice (0.123 to 0.136, 7 of 12 higher) is
not significant (p = 0.20), so "K higher in both DREADD groups" fails; the criterion names a
partial outcome only for the case where K and AUC hold and A does not. Reading (b), Friedman,
which with two conditions reduces to a sign test, gives the same pattern with fewer significant
results (hM4Di AUC p = 0.15).

## Repeated dosing: fig6b-ctrl-ppx-stable, fig6c-park-ppx-progressive, fig6de-a77636-persistent

Split-plot ANOVA over baseline, first and fourth ON session, with Bonferroni comparisons of the
condition means pooled over delays (and per delay as a variant).

In [ ]:
def sessions_test(exp, group):
    base = per_mouse(exp, group, "baseline", "all")
    first = per_mouse(exp, group, "on", "first")
    fourth = per_mouse(exp, group, "on", "fourth")
    L = pd.concat([long(base, t="baseline"), long(first, t="first"), long(fourth, t="fourth")])
    res = icd.split_plot_anova(L, "v", "t", ["baseline", "first", "fourth"])
    pairs = icd.main_effect_pairs(L, "v", "t", ["baseline", "first", "fourth"])
    print(f"{exp} {group}: mice with a fourth session {len(fourth)} of {len(base)}; "
          f"session F{res['treat_df']} = {res['treat_F']:.2f}, p = {res['treat_p']:.3g}")
    print(pairs.round(4).to_string(index=False))
    L2 = pd.concat([long(first, t="first"), long(fourth, t="fourth")])
    print(icd.split_plot_anova(L2, "v", "t", ["first", "fourth"])["per_delay"].round(4).to_string(index=False))
    return pairs

In [ ]:
sessions_test("PPX", "control")

**Result: reproduced.** In control mice no pair of baseline, first and fourth PPX session
differs (Bonferroni p = 0.46, 0.75 and 1; paper first against fourth p > 0.99); 12 of 16 control
mice have a fourth PPX session.

In [ ]:
sessions_test("PPX", "parkinsonian")

**Result: reproduced.** In the 27 parkinsonian mice with four PPX sessions, the
delayed/large choices fall from 50.7% at baseline to 41.6% in the first and 36.2% in the fourth
session; baseline differs from both (p = 2.7e-05, 2.7e-11) and the fourth from the first
(p = 0.024; paper p < 0.0081).

In [ ]:
sessions_test("A77636", "parkinsonian")
sessions_test("A77636", "control")

**Result: reproduced.** In parkinsonian mice baseline differs from the first and fourth
A77636 session (p = 0.0004 and < 0.0001) and the two sessions do not differ (p = 1; paper 0.59). In
controls no comparison is significant (smallest p = 0.31; paper first against fourth 0.49).

## fig1b-th-loss

> 6-OHDA caused about 50% loss of TH immunofluorescence in the rostral dorsolateral and
> dorsomedial striatum, with less depletion in the ventral striatum.

CTF per ROI (Supplementary methods), normalized to the mean of the control mice of the same staining
cohort (each cohort "includes both healthy and parkinsonian samples to control for inter-batch
staining variability", deposit README); reading (b) normalizes to the mean of all control mice.
Kruskal-Wallis over DLS, DMS and VS of the parkinsonian mice with Dunn's comparisons.

In [ ]:
sec = icd.striatal_th()
print("sections:", len(sec), "| mice per cohort and group:")
print(sec.groupby(["cohort", "group"]).mouse.nunique().unstack().to_string())
M = sec.groupby(["cohort", "group", "mouse"])[["DLS", "DMS", "VS"]].mean().reset_index()
ctrl_cohort = M[M.group == "control"].groupby("cohort")[["DLS", "DMS", "VS"]].mean()
ctrl_all = M[M.group == "control"][["DLS", "DMS", "VS"]].mean()

def dunn_kw(df, cols):
    allv = pd.concat([df[c] for c in cols]).rank()
    n = len(df)
    ranks = {c: allv.iloc[i * n:(i + 1) * n].mean() for i, c in enumerate(cols)}
    N = n * len(cols)
    se = np.sqrt(N * (N + 1) / 12 * (2 / n))
    pairs = [(a, b) for i, a in enumerate(cols) for b in cols[i + 1:]]
    return {f"{a} vs {b}": min(1, 2 * stats.norm.sf(abs(ranks[a] - ranks[b]) / se) * len(pairs)) for a, b in pairs}

th = {}
for reading, ref in [("per cohort", None), ("all controls", ctrl_all)]:
    N = M.copy()
    for r in ["DLS", "DMS", "VS"]:
        N[r] = N[r] / (N.cohort.map(ctrl_cohort[r]) if ref is None else ref[r])
    P = N[N.group == "parkinsonian"].groupby("mouse")[["DLS", "DMS", "VS"]].mean()
    th[reading] = P
    loss = 100 * (1 - P.mean())
    print(f"{reading}: {len(P)} parkinsonian mice; loss % mean {loss.round(1).to_dict()}, "
          f"median {(100 * (1 - P.median())).round(1).to_dict()}; Kruskal-Wallis p = {stats.kruskal(P.DLS, P.DMS, P.VS).pvalue:.3g}; "
          f"Dunn {dict((k, round(v, 4)) for k, v in dunn_kw(P, ['DLS', 'DMS', 'VS']).items())}")

**Result: partially reproduced.** VS is less depleted than DLS and DMS and DLS does not
differ from DMS under the per-cohort reading (Dunn p < 0.0001, 0.0001 and 0.38), but the mean losses
in DLS (64.3%) and DMS (56.8%) are outside 50 +/- 5%. Normalized to all controls, the DMS loss
(52.4%) is inside the tolerance and DLS (60.5%) is not, and VS against DMS gives p = 0.052. 45
parkinsonian mice enter (Table 1 gives N = 44).

## figs1d-snc-cells

> Parkinsonian mice had fewer TH-positive dopamine neurons in the substantia nigra pars compacta
> than controls.

Density = TH+ cells counted over the four sections and both sides, divided by the area counted.
The workbook has no group column. Groups come, in this order, from the histology workbook of the same
deposit (its "Ctrl" and "Park" blocks), the deposit README's PPX lists, and, for the
electrophysiology mice, the DANDI subjects' lesion (saline or 6-OHDA).

In [ ]:
snc = icd.read_excel(icd.INSTRUMENTAL)["Fig. S1D"].iloc[1:].copy()
snc.columns = ["ap", "image", "notes", "mouse", "lc", "la", "rc", "ra"]
for c in ["lc", "la", "rc", "ra"]:
    snc[c] = pd.to_numeric(snc[c], errors="coerce")
bad = snc[(snc.la < 1000) | (snc.ra < 1000)]
print("rows whose area is under 1000 (columns shifted):", bad[["mouse", "lc", "la", "rc", "ra"]].values.tolist())
snc.loc[bad.index, ["lc", "la", "rc", "ra"]] = np.nan
mouse_n = snc.mouse.str.replace("JF-", "JF").str.strip()
per = snc.assign(mouse=mouse_n).groupby("mouse").apply(
    lambda g: pd.Series({"cells": g.lc.sum() + g.rc.sum(), "area": g.la.sum() + g.ra.sum()}))
per["density"] = per.cells / per.area * 1e6
ppx_c, ppx_p = groups[("PPX", "control")], groups[("PPX", "parkinsonian")]
dandi_lesion = {s.replace("-", "."): l for s, l in zip(subj.subject, subj.lesion)}
dandi_lesion.update({s.replace("-", ""): l for s, l in zip(subj.subject, subj.lesion)})
histo_group = icd.striatal_th().drop_duplicates("mouse").set_index("mouse").group.to_dict()
def snc_group(m):
    if m in histo_group:
        return histo_group[m]
    if m in ppx_c:
        return "control"
    if m in ppx_p:
        return "parkinsonian"
    return {"saline": "control", "6-OHDA": "parkinsonian"}.get(dandi_lesion.get(m), None)
per["group"] = [snc_group(m) for m in per.index]
print(per.round(2).to_string())
lab = per.dropna(subset=["group"])
c, p = lab[lab.group == "control"].density, lab[lab.group == "parkinsonian"].density
print(f"linked mice: {len(lab)} of {len(per)} (control {len(c)}, parkinsonian {len(p)}); median density "
      f"control {c.median():.1f}, parkinsonian {p.median():.1f} cells/mm2 (area units assumed um2); "
      f"Mann-Whitney p = {stats.mannwhitneyu(c, p).pvalue:.3g}")
unl = per[per.group.isna()].density
for g in ["control", "parkinsonian"]:
    cc = pd.concat([c, unl]) if g == "control" else c
    pp = pd.concat([p, unl]) if g == "parkinsonian" else p
    print(f"   unlinked mice all counted as {g}: Mann-Whitney p = {stats.mannwhitneyu(cc, pp).pvalue:.3g}")

**Result: reproduced.** In the 29 mice whose group is known (9 control, 20 parkinsonian; the
paper has 9 and 21), TH+ cell density in the SNc is lower in parkinsonian mice (median 186.6
against 316.0 cells per mm2, Mann-Whitney p = 0.0003). The three mice in no list (C0.1, C2.1,
D5.2) do not change this whichever group they are put in (p = 0.0001 and 0.0019). Two rows of the
workbook (D3 and N4 at -3.51 mm) have their count and area columns shifted and are left out.

## fig1c-rotarod

> Parkinsonian mice fell sooner from the accelerating rotarod than controls, and PPX improved their
> performance.

In [ ]:
rr = icd.read_excel("Data_ICD_Fig1_Rotarod_PPX.xlsx")["Sheet1"]
cols = {rr.iat[0, j]: pd.to_numeric(rr.iloc[2:, j], errors="coerce").dropna().reset_index(drop=True)
        for j in range(rr.shape[1]) if isinstance(rr.iat[0, j], str)}
for k, v in cols.items():
    print(f"{k}: n = {len(v)}, median {v.median():.1f} s, range {v.min():.1f} to {v.max():.1f}")
ctrl = cols["Ctrl_baseline"]
p1, ppx = cols["Park_cohort1_baseline"], cols["Park_cohort1_PPX"]
p2, sal = cols["Park_cohort2_baseline"], cols["Park_cohort2_Saline"]
kw = stats.kruskal(ctrl, p1, ppx, p2, sal)
allv = pd.concat([ctrl, p1, ppx, p2, sal], keys=["ctrl", "p1", "ppx", "p2", "sal"]).rank()
N = len(allv)
def dunn(a, b, m=3):
    ra, rb = allv[a].mean(), allv[b].mean()
    se = np.sqrt(N * (N + 1) / 12 * (1 / len(allv[a]) + 1 / len(allv[b])))
    return min(1, 2 * stats.norm.sf(abs(ra - rb) / se) * m)
print(f"Kruskal-Wallis over the five columns p = {kw.pvalue:.3g}; Dunn (3 comparisons): control vs park "
      f"baseline p = {dunn('ctrl', 'p1'):.3g}, park baseline vs PPX p = {dunn('p1', 'ppx'):.3g}, "
      f"park baseline vs saline p = {dunn('p2', 'sal'):.3g}")
print(f"variant: Mann-Whitney control vs park baseline (cohort 1) p = {stats.mannwhitneyu(ctrl, p1).pvalue:.3g}; "
      f"Wilcoxon baseline vs PPX p = {stats.wilcoxon(p1, ppx).pvalue:.3g}; baseline vs saline p = {stats.wilcoxon(p2, sal).pvalue:.3g}")

**Result: reproduced.** Parkinsonian mice fall sooner than controls (median 47.3 against
91.3 s, Dunn p = 1.1e-15), PPX raises their latency (to 66.7 s, Dunn p = 1.8e-05) and saline does
not (Dunn p = 0.83). The variant tests agree (Mann-Whitney p = 5e-13, Wilcoxon p = 3e-08 and
0.074). The workbook has 40 control, 45 PPX and 27 saline mice; Table 1 gives 31, 45 and 27.

## figs1-instrumental-learning

> Parkinsonian mice had slightly slower response latencies and learning in the first instrumental
> learning phase but reached performance similar to controls in the second.

Mann-Whitney per session, Bonferroni over six sessions, from the workbook's Figure S1H-I and S1K-L
sheets. The second "Session1_Park" header of the S1K-L sheet (row 118), between session 1
parkinsonian and session 2 parkinsonian, is read as session 2 control.

In [ ]:
inst = icd.instrumental_tables()
fix = (inst.phase == "random") & (inst.header_row == 118)
inst.loc[fix, ["session", "group"]] = [2, "control"]
rows = []
for (phase, s), g in inst.groupby(["phase", "session"]):
    c, p = g[g.group == "control"], g[g.group == "parkinsonian"]
    for m in ["success", "latency"]:
        rows.append({"phase": phase, "session": s, "measure": m, "n_ctrl": c[m].notna().sum(),
                     "n_park": p[m].notna().sum(), "median_ctrl": c[m].median(), "median_park": p[m].median(),
                     "p_bonf": min(1, stats.mannwhitneyu(c[m].dropna(), p[m].dropna()).pvalue * 6)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: partially reproduced.** In blocked learning parkinsonian mice have lower success
in sessions 1, 3 and 4 (Bonferroni p = 0.007, 0.0055, 0.0035) and longer latencies in sessions 3
to 6 (p = 0.0005 to 0.012). In random learning they are similar in sessions 2 to 6, but session 1
success differs (65.9 against 86.9%, p = 0.003), against the criterion's "no session of random
learning differs". Initiation latency differs in no random session.

## Correlations with striatal TH: fig3a-a-correlates-dms, fig3b-k-no-correlation, figs4-a77636-no-correlation

A and K on the drug (four ON sessions) of each parkinsonian mouse against its normalized TH
(per-cohort reading of fig1b-th-loss), Pearson correlation.

In [ ]:
def corr_block(exp, label):
    on = params(per_mouse(exp, "parkinsonian", "on", "four"))
    out = {}
    for reading, P in th.items():
        j = on.join(P, how="inner")
        for p in ["A", "K"]:
            for r in ["DMS", "DLS", "VS"]:
                rr_, pp = stats.pearsonr(j[p], j[r])
                out[(reading, p, r)] = (len(j), rr_, pp)
                print(f"{label} {reading}: {p} vs {r} TH, n = {len(j)}, r = {rr_:.2f}, p = {pp:.3g}")
    return out
corr_ppx = corr_block("PPX", "PPX")

**Result: not reproduced.** A on PPX correlates with DMS TH at r = 0.32, p = 0.13 in the
24 mice with both measures (paper r = 0.51, p = 0.006, n = 28), outside the tolerance (0.46 to
0.56); DLS and VS are not significant (p = 0.29 and 0.18). Normalized to all controls, r = 0.14.

In [ ]:
for (reading, p, r), (n, rr_, pp) in corr_ppx.items():
    if p == "K":
        print(f"PPX {reading}: K vs {r}, n = {n}, r = {rr_:.2f}, p = {pp:.3g}")

**Result: reproduced.** K on PPX does not correlate with TH in any subregion (DMS r = 0.03,
DLS r = 0.40, VS r = 0.32; p = 0.89, 0.056, 0.13; n = 24).

In [ ]:
corr_a77 = corr_block("A77636", "A77636")

**Result: reproduced.** None of the six correlations of A and K on A77636 with TH is
significant (smallest p = 0.24, K against VS, r = -0.31; paper r = -0.31, p = 0.23; n = 16).

## figs6-dreadd-validation

> In slices CNO reduced optically evoked IPSCs from hM4Di-expressing iMSNs and increased the firing
> of hM3Dq-expressing dMSNs.

In [ ]:
sl = icd.read_excel("Data_ICD_FigS6_SliceValidation.xlsx")["Sheet1"]
oipsc = sl.iloc[1:7, 1:3].apply(pd.to_numeric, errors="coerce")
oipsc.columns = ["baseline", "cno"]
print(oipsc.to_string())
print(f"oIPSC: n = {len(oipsc)}, median {oipsc.baseline.median():.1f} -> {oipsc.cno.median():.1f}, "
      f"Wilcoxon p = {stats.wilcoxon(oipsc.baseline, oipsc.cno).pvalue:.3g}")
def io_block(r0):
    blk = sl.iloc[r0:r0 + 6, 1:11].apply(pd.to_numeric, errors="coerce")
    blk.index = blk.iloc[:, 0].astype(int)
    return blk.iloc[:, 1:]
base_io, cno_io = io_block(10), io_block(21)
print("cells:", base_io.shape[1])
print(f"firing at 300 pA: median {base_io.loc[300].median():.1f} -> {cno_io.loc[300].median():.1f} Hz, "
      f"Wilcoxon p = {stats.wilcoxon(base_io.loc[300], cno_io.loc[300]).pvalue:.3g}")
Lio = pd.concat([base_io.loc[[100, 200, 300, 400]].T.reset_index(drop=True).rename_axis("mouse").reset_index()
                 .melt(id_vars="mouse", var_name="delay", value_name="v").assign(t="baseline"),
                 cno_io.loc[[100, 200, 300, 400]].T.reset_index(drop=True).rename_axis("mouse").reset_index()
                 .melt(id_vars="mouse", var_name="delay", value_name="v").assign(t="cno")])
res = icd.split_plot_anova(Lio, "v", "t", ["baseline", "cno"])
print(f"input-output, CNO effect (cell x current units): F{res['treat_df']} = {res['treat_F']:.2f}, p = {res['treat_p']:.3g}")
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
fit = smf.ols("v ~ C(t) * C(delay)", data=Lio).fit()
print("ordinary two-way ANOVA (variant):", anova_lm(fit, typ=2).round(4).to_dict()["PR(>F)"], "error df", int(fit.df_resid))

**Result: reproduced.** CNO reduces oIPSC amplitude (median 647.5 to 55.0 pA, Wilcoxon
p = 0.031, 6 cells) and raises firing at 300 pA (8 to 16 Hz, p = 0.016, 9 cells), and the CNO
effect on the input-output curve is F(1, 32) = 17.64, p = 0.0002, the paper's values, when each
cell and current is the repeated unit; an ordinary two-way ANOVA gives p = 0.082.

## Open field: fig2bc-a77636-locomotion and fig5cf-cno-locomotion

The files hold time (s) and velocity (cm/s) at 15 frames/s and no injection mark. Nearly all have
a short run of untracked frames at 20.0 to 20.7 min, consistent with the mouse being lifted for the
injection after a 20 min baseline. Injection is taken at 20 min (an assumption); a -30 to 0 min
baseline does not exist in these files, so only the -20 to 0 min reading can be run.

In [ ]:
def of_table(zip_name, pre, post):
    of = icd.open_field(zip_name)
    rows = []
    for _, r in of.iterrows():
        t, v = r.t_min - 20, r.velocity
        gaps = icd.tracking_gaps(r.t_min, v)
        rows.append({"mouse": r.mouse, "file": r.file, "end_min": r.t_min[-1],
                     "gap_near_20": f"{gaps[0][0]:.1f}" if gaps else "",
                     "pre": np.nanmean(v[(t >= pre[0]) & (t < pre[1])]),
                     "post": np.nanmean(v[(t >= post[0]) & (t < post[1])]),
                     "post_covered": r.t_min[-1] - 20 >= post[1]})
    return pd.DataFrame(rows)
a77_of = of_table("Data_ICD_A77636OpenField_Raw.zip", (-20, 0), (30, 100))
ctrl_ids = groups[("A77636", "control")]
a77_of["group"] = np.where(a77_of.mouse.isin(ctrl_ids), "control", "not in the control list")
print(a77_of.round(2).to_string(index=False))
for g, t in a77_of.groupby("group"):
    print(f"{g}: n = {len(t)}, median {t.pre.median():.2f} -> {t.post.median():.2f} cm/s, "
          f"Wilcoxon p = {stats.wilcoxon(t.pre, t.post).pvalue:.3g}, faster in {int((t.post > t.pre).sum())}")
print("mice not in either A77636 list:", sorted(set(a77_of.mouse) - set(ctrl_ids) - set(groups[("A77636", "parkinsonian")])))

**Result: reproduced, resting on an assumption.** With injection at 20 min, the 16 mice
not in the control list are faster 30 to 100 min after A77636 (median 3.14 to 3.69 cm/s, 13 of
16, Wilcoxon p = 0.016; paper p = 0.02) and the 10 controls are not (3.19 to 2.90 cm/s, p = 0.065;
paper 0.16). Six of the 16 (O6, R0, R1, R3.2, R5.3, R6) are in neither A77636 list; they are taken
as parkinsonian because the control list is complete (10 of 10) and 16 mice remain, the paper's
N. R1.1's file has long tracking losses before 20 min and its first gap near the injection at 15 min.

In [ ]:
cno_of = of_table("Data_ICD_Fig5_DREADDLocomotion_Raw.zip", (-20, 0), (20, 30))
def dgroup(m):
    for g in ["mCherry", "hM4Di", "hM3Dq"]:
        if m in groups[("DREADD", g)]:
            return g
    return None
cno_of["group"] = cno_of.mouse.map(dgroup)
print(cno_of.round(2).to_string(index=False))
for g, t in cno_of.dropna(subset=["group"]).groupby("group"):
    print(f"{g}: n = {len(t)}, median {t.pre.median():.2f} -> {t.post.median():.2f} cm/s, "
          f"Wilcoxon p = {stats.wilcoxon(t.pre, t.post).pvalue:.3g}, faster in {int((t.post > t.pre).sum())}")
print("files whose mouse is in no DREADD list:", cno_of[cno_of.group.isna()].file.tolist())

**Result: partially reproduced, resting on an assumption.** With injection at 20 min,
hM3Dq mice (11 of 11 faster, p = 0.001) and hM4Di mice (9 of 10, p = 0.004) move faster 20 to 30 min
after CNO, but mCherry mice move slower (2.96 to 2.23 cm/s, 1 of 12 faster, p = 0.016), where the
criterion needs p >= 0.05. Four files (R0, F0, F3, R1.2) are of mice in no DREADD list, and most
2023 files (H mice) have no tracking gap near 20 min, so their injection time rests on the assumption alone.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| fig1b-th-loss | Zenodo | partially reproduced (pattern holds; losses 64 and 57% against 50 +/- 5%) |
| figs1d-snc-cells | Zenodo | reproduced |
| fig1c-rotarod | Zenodo | reproduced |
| figs1ef-ppx-locomotion | Zenodo | blocked: no PPX open field file in the deposit |
| figs1-instrumental-learning | Zenodo | partially reproduced (random learning session 1 differs) |
| fig1fg-baseline-similar | Zenodo | reproduced |
| fig1fg-delay-discounting | Zenodo | reproduced |
| fig1f-ctrl-ppx-no-change | Zenodo | reproduced |
| fig1g-park-ppx-fewer-large | Zenodo | reproduced |
| fig1ijk-ppx-a-k-auc | Zenodo | partially reproduced (K rise not significant by Dunn's test) |
| figs2abc-sex-similar | Zenodo | blocked: no sex per mouse in the deposit |
| figs2de-omissions-low | Zenodo | reproduced |
| figs2gh-ppx-latency | Zenodo | partially reproduced (PPX shortening of small-reward latency not significant) |
| figs2ijk-ppx-washout | Zenodo | partially reproduced (A recovers, K or AUC persists depending on the test) |
| fig2bc-a77636-locomotion | Zenodo | reproduced, resting on an assumption (injection at 20 min) |
| fig2d-ctrl-a77636-no-change | Zenodo | reproduced |
| fig2e-park-a77636-fewer-large | Zenodo | reproduced |
| fig2fgh-a77636-k-auc | Zenodo | reproduced |
| figs3-a77636-washout-latency | Zenodo | partially reproduced (reward size effect on latency remains on A77636) |
| fig3a-a-correlates-dms | Zenodo | not reproduced (r = 0.32, p = 0.13) |
| fig3b-k-no-correlation | Zenodo | reproduced |
| figs4-a77636-no-correlation | Zenodo | reproduced |
| figs6-dreadd-validation | Zenodo | reproduced |
| fig5cf-cno-locomotion | Zenodo | partially reproduced, resting on an assumption (mCherry mice slower) |
| fig5g-mcherry-no-change | Zenodo | reproduced |
| fig5hi-dreadd-fewer-large | Zenodo | partially reproduced (hM4Di significant at 3 s only) |
| fig5jl-dreadd-params | Zenodo | partially reproduced (hM4Di K rise not significant) |
| fig6b-ctrl-ppx-stable | Zenodo | reproduced |
| fig6c-park-ppx-progressive | Zenodo | reproduced |
| fig6de-a77636-persistent | Zenodo | reproduced |
| fig4bc, fig4de (2), fig4f, fig4g, fig4 insets, fig4hk, figs5abcd, figs5ef, fig6f, fig6g, figs8, fig6h, fig6i (14 claims) | DANDI:001177 | blocked: data missing |

How the analysis differs from the paper's. The paper's per-mouse spreadsheets of choices were not
deposited, so every session is recounted from the raw operant logs with the paper's event names.
The baseline sessions are every baseline session deposited (2 to 5 per mouse), since the CV rule
finds a stable run in only 12 of 47 PPX mice. The "two-way repeated measures ANOVA" is run with
each mouse and delay as the unit, the layout the paper's degrees of freedom imply. The open field
windows assume injection at 20 min. Striatal TH is normalized to the control mice of each staining
cohort.

What was found about the data.

- DANDI:001177 holds raw voltage, LFP and per-electrode threshold crossings for 18 mice and 56
  sessions, matching the paper's mice (7 control, 11 parkinsonian, 4 with saline sessions), but no
  sorted units in 54 files, no unit labels in the other two, no laser pulse times and no injection
  time, so none of the electrophysiology findings can be tested. Three sessions have start times
  in 2025, out of order with the same mouse's other days, and the subject descriptions are
  mis-encoded ("1.5â€¯microliters").
- In the Zenodo deposit, 233 operant logs are byte-identical between the PPX and DREADD folders
  and 75 between the A77636 and DREADD folders (mice used in two experiments); one PPX
  parkinsonian mouse (E6) has no PPX sessions and one mCherry mouse (R5.5) no delay discounting
  files; the 2020 control cohort ran delays of 0, 6, 12 and 18 s and the 2021 cohort 30 free
  choice trials per block, unlike the Methods; the open field files carry no injection time; no
  file gives each mouse's sex; and no PPX open field file is deposited.